# 31 · ADK 2 graphs and typed data

Graph workflows combine deterministic Python steps and agents. Start with a zero-model invoice flow, then add a model only where language generation helps.

**Runtime:** ADK 2.10.0 on `aadhi`, existing `dataengenv` kernel and ADC.

**Cost:** Zero model calls. Synthetic ecommerce data only. Run cells from top to bottom; each notebook creates its own sessions.


In [ ]:
from pathlib import Path
import sys
# Works when Jupyter starts in GoogleADK or its parent folder.
root = Path.cwd() if (Path.cwd() / "course.py").exists() else Path.cwd() / "GoogleADK"
if not (root / "course.py").exists():
    raise RuntimeError("Open this notebook with GoogleADK as the working directory.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from course import (ROOT, CONFIG, MODEL, PROJECT, LOCATION, SKIP_SSL, DATA,
                    llm, gemini, make_lab, ask, final_text, token_usage,
                    get_order, get_product, get_customer, types)
print(f"Project={PROJECT} | location={LOCATION} | model={MODEL} | TLS bypass={SKIP_SSL}")


In [ ]:
from google.adk import Workflow, Event
from pydantic import BaseModel

class OrderFacts(BaseModel):
    order_id: str
    status: str
    total: float

def fetch_order(node_input: str) -> OrderFacts:
    order = get_order(node_input.strip())
    if "error" in order:
        raise ValueError(order["error"])
    return OrderFacts(order_id=order["order_id"], status=order["status"], total=order["total"])

def render_receipt(node_input: OrderFacts):
    return Event(message=f"{node_input.order_id}: {node_input.status}; total INR {node_input.total:.2f}")

workflow = Workflow(name="invoice_graph", edges=[("START", fetch_order, render_receipt)])
lab = await make_lab(workflow)
await ask(lab, "O1001")


In [ ]:
def classify_order(node_input: str):
    order = get_order(node_input.strip())
    return Event(route="delivered" if order.get("status") == "delivered" else "other")

def return_options():
    return Event(message="Check delivery age and membership before approving any return.")

def processing_options():
    return Event(message="Check fulfillment status before quoting a dispatch date.")

routing = Workflow(name="order_routes", edges=[("START", classify_order),
    (classify_order, {"delivered": return_options, "other": processing_options})])
route_lab = await make_lab(routing)
await ask(route_lab, "O1001")


## Try it

Add an explicit unknown-order route, then replace a final renderer with an Agent using input_schema=OrderFacts.


## Reference

[Official ADK documentation](https://adk.dev/graphs/). Shared configuration: `config.json`. No environment activation or login cells are needed.
